# AWS Bedrock: Multi-Tool Presentation Agent with Web Research & Image Generation

This notebook demonstrates how to build an **advanced presentation agent** using the LangChain **`create_agent`** function and LangGraph **`MemorySaver`** persistence.

The agent is powered by **Claude 3.5 Haiku** (`us.anthropic.claude-haiku-4-5-20251001-v1:0` via AWS Bedrock) and has access to three tools:
1. **`create_premium_presentation`**: Builds widescreen dark-themed slides, automatically applying a predefined background template image (`skills/ppt_design/resources/slide_background_theme.png`) for title/ending layouts.
2. **`generate_slide_image`**: Calls Bedrock's `amazon.titan-image-generator-v1` model to generate custom visual assets for specific slide illustrations.
3. **`duckduckgo_search`**: Searches the web to research the user's requested topic and verify technical details before draft creation.

### Prerequisites
1. Install packages: `pip install langchain-aws langchain langgraph python-pptx boto3 langchain-community duckduckgo-search`
2. Ensure AWS credentials are configured on your local machine.

## Step 1: Install Dependencies

Uncomment and run the cell below to install the required libraries.

In [1]:
# !pip install -U langchain-aws langchain langgraph python-pptx boto3 langchain-community duckduckgo-search

## Step 2: Define PowerPoint and Image Generation Tools

We define the `create_premium_presentation` tool alongside a new tool `generate_slide_image` which calls the Bedrock Titan Image Generator model to compile custom slide assets.

In [2]:
import os
import json
import base64
import boto3
import threading
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN
from pptx.enum.shapes import MSO_SHAPE
from langchain_core.tools import tool

# Global lock to serialize image generation requests and prevent Bedrock throttling
image_lock = threading.Lock()

@tool
def create_premium_presentation(topic: str, slides: list[dict], output_file: str = "generated_presentation.pptx") -> str:
    """
    Creates a premium, widescreen (16:9) dark-themed PowerPoint presentation on a given topic.
    
    Args:
        topic (str): The overall topic of the presentation.
        slides (list[dict]): A list of dictionaries representing slides. Each slide dictionary must contain:
            - 'section': Section category label (e.g., 'AWS Bedrock Overview')
            - 'title': Slide title (e.g., 'Key Features')
            - 'content': List of strings (bullet points)
            - 'layout': Layout type ('title', 'text', 'image', or 'ending')
            - 'image_path': (Optional) Local path to an image to embed on the slide (e.g., 'slide_illustration.png').
        output_file (str): The filename for the output PPTX file.
        
    Returns:
        str: A success message with the absolute file path.
    """
    try:
        prs = Presentation()
        prs.slide_width = Inches(13.333)
        prs.slide_height = Inches(7.5)
        blank_layout = prs.slide_layouts[6]

        # Predefined Background Image Resource
        bg_image_path = "skills/ppt_design/resources/slide_background_theme.png"
        if not os.path.exists(bg_image_path):
            bg_image_path = "experiment/skills/ppt_design/resources/slide_background_theme.png"
        
        # Theme Colors
        BG_COLOR = RGBColor(11, 15, 25)       # Deep Slate Blue
        TEXT_WHITE = RGBColor(248, 250, 252)  # Snow White
        TEXT_MUTED = RGBColor(148, 163, 184)  # Muted Blueish Gray
        ACCENT_CYAN = RGBColor(0, 242, 254)   # Neon Cyan
        ACCENT_PURPLE = RGBColor(138, 43, 226) # Royal Purple
        CARD_BG = RGBColor(22, 30, 49)        # Lighter Card Slate
        BORDER_COLOR = RGBColor(40, 55, 87)   # Muted Border

        for i, slide_data in enumerate(slides):
            slide = prs.slides.add_slide(blank_layout)
            layout_type = slide_data.get("layout", "text")
            
            # Apply full-bleed background for title/ending slides
            if (layout_type == "title" or layout_type == "ending") and os.path.exists(bg_image_path):
                slide.shapes.add_picture(bg_image_path, Inches(0), Inches(0), Inches(13.333), Inches(7.5))
            else:
                bg = slide.background
                fill = bg.fill
                fill.solid()
                fill.fore_color.rgb = BG_COLOR
            
            # Title Layout
            if layout_type == "title":
                title_box = slide.shapes.add_textbox(Inches(1.0), Inches(2.2), Inches(11.333), Inches(3.5))
                tf = title_box.text_frame
                tf.word_wrap = True
                
                p_lbl = tf.paragraphs[0]
                p_lbl.text = slide_data.get("section", "PRESENTATION").upper()
                p_lbl.font.name = 'Arial'
                p_lbl.font.size = Pt(11)
                p_lbl.font.bold = True
                p_lbl.font.color.rgb = ACCENT_CYAN
                p_lbl.alignment = PP_ALIGN.CENTER
                p_lbl.space_after = Pt(12)
                
                p_title = tf.add_paragraph()
                p_title.text = slide_data.get("title", topic).upper()
                p_title.font.name = 'Arial'
                p_title.font.size = Pt(36)
                p_title.font.bold = True
                p_title.font.color.rgb = TEXT_WHITE
                p_title.alignment = PP_ALIGN.CENTER
                
                content_list = slide_data.get("content", [])
                if content_list:
                    p_sub = tf.add_paragraph()
                    p_sub.text = content_list[0]
                    p_sub.font.name = 'Arial'
                    p_sub.font.size = Pt(16)
                    p_sub.font.color.rgb = TEXT_MUTED
                    p_sub.alignment = PP_ALIGN.CENTER
                    p_sub.space_before = Pt(20)
            
            # Ending Layout
            elif layout_type == "ending":
                txBox = slide.shapes.add_textbox(Inches(1.0), Inches(2.0), Inches(6.0), Inches(4.5))
                tf = txBox.text_frame
                tf.word_wrap = True
                p_lbl = tf.paragraphs[0]
                p_lbl.text = "SUMMARY & DISCUSSION"
                p_lbl.font.name = 'Arial'
                p_lbl.font.size = Pt(11)
                p_lbl.font.bold = True
                p_lbl.font.color.rgb = ACCENT_CYAN
                p_lbl.space_after = Pt(12)
                
                content_list = slide_data.get("content", [])
                for bullet in content_list:
                    p = tf.add_paragraph()
                    p.text = "• " + bullet
                    p.font.name = 'Arial'
                    p.font.size = Pt(13)
                    p.font.color.rgb = TEXT_MUTED
                    p.space_before = Pt(8)
                
                card = slide.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE, Inches(7.5), Inches(2.0), Inches(5.083), Inches(4.0))
                card.fill.solid()
                card.fill.fore_color.rgb = CARD_BG
                card.line.color.rgb = BORDER_COLOR
                card.line.width = Pt(1.5)
                tf_card = card.text_frame
                tf_card.word_wrap = True
                tf_card.margin_left = tf_card.margin_right = tf_card.margin_top = tf_card.margin_bottom = Inches(0.2)
                
                p_c = tf_card.paragraphs[0]
                p_c.text = "Ready for Labs"
                p_c.font.name = 'Arial'
                p_c.font.size = Pt(14)
                p_c.font.bold = True
                p_c.font.color.rgb = ACCENT_PURPLE
                p_c.space_after = Pt(10)
                p_cd = tf_card.add_paragraph()
                p_cd.text = "Review loaded notebooks in your workspace. Test code inference endpoints, verify cross-region configurations, and execute agent runs."
                p_cd.font.name = 'Arial'
                p_cd.font.size = Pt(11)
                p_cd.font.color.rgb = TEXT_WHITE

            # Content Layouts
            else:
                lbl_box = slide.shapes.add_textbox(Inches(0.75), Inches(0.4), Inches(11.833), Inches(0.3))
                tf_lbl = lbl_box.text_frame
                tf_lbl.word_wrap = True
                p_lbl = tf_lbl.paragraphs[0]
                p_lbl.text = slide_data.get("section", "").upper()
                p_lbl.font.name = 'Arial'
                p_lbl.font.size = Pt(10)
                p_lbl.font.bold = True
                p_lbl.font.color.rgb = ACCENT_CYAN
                
                title_box = slide.shapes.add_textbox(Inches(0.75), Inches(0.7), Inches(11.833), Inches(0.7))
                tf_title = title_box.text_frame
                tf_title.word_wrap = True
                p_title = tf_title.paragraphs[0]
                p_title.text = slide_data.get("title", "")
                p_title.font.name = 'Arial'
                p_title.font.size = Pt(26)
                p_title.font.bold = True
                p_title.font.color.rgb = TEXT_WHITE

                content_list = slide_data.get("content", [])
                image_path = slide_data.get("image_path", "")
                
                if layout_type == "image" and image_path and os.path.exists(image_path):
                    txBox = slide.shapes.add_textbox(Inches(0.75), Inches(1.8), Inches(5.5), Inches(4.5))
                    tf_tx = txBox.text_frame
                    tf_tx.word_wrap = True
                    for bullet in content_list:
                        p = tf_tx.add_paragraph() if tf_tx.text else tf_tx.paragraphs[0]
                        p.text = "• " + bullet
                        p.font.name = 'Arial'
                        p.font.size = Pt(13)
                        p.font.color.rgb = TEXT_MUTED
                        p.space_before = Pt(8)
                    
                    slide.shapes.add_picture(image_path, Inches(6.75), Inches(1.8), Inches(5.833), Inches(3.8))
                else:
                    txBox = slide.shapes.add_textbox(Inches(0.75), Inches(1.8), Inches(6.0), Inches(4.5))
                    tf_tx = txBox.text_frame
                    tf_tx.word_wrap = True
                    for bullet in content_list:
                        p = tf_tx.add_paragraph() if tf_tx.text else tf_tx.paragraphs[0]
                        p.text = "• " + bullet
                        p.font.name = 'Arial'
                        p.font.size = Pt(13)
                        p.font.color.rgb = TEXT_MUTED
                        p.space_before = Pt(8)
                    
                    card = slide.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE, Inches(7.5), Inches(1.8), Inches(5.083), Inches(4.5))
                    card.fill.solid()
                    card.fill.fore_color.rgb = CARD_BG
                    card.line.color.rgb = BORDER_COLOR
                    card.line.width = Pt(1.5)
                    tf_card = card.text_frame
                    tf_card.word_wrap = True
                    tf_card.margin_left = tf_card.margin_right = tf_card.margin_top = tf_card.margin_bottom = Inches(0.2)
                    
                    p_c = tf_card.paragraphs[0]
                    p_c.text = "Key Insight:"
                    p_c.font.name = 'Arial'
                    p_c.font.size = Pt(14)
                    p_c.font.bold = True
                    p_c.font.color.rgb = ACCENT_PURPLE
                    p_c.space_after = Pt(10)
                    p_cd = tf_card.add_paragraph()
                    p_cd.text = f"This slide covers '{slide_data.get('title')}' in context of {topic}. Focus on discussing design boundaries and parameters during lecture."
                    p_cd.font.name = 'Arial'
                    p_cd.font.size = Pt(11)
                    p_cd.font.color.rgb = TEXT_MUTED

        output_path = os.path.abspath(output_file)
        prs.save(output_path)
        return f"Successfully created presentation at: {output_path}"
    except Exception as e:
        return f"Failed to create presentation. Error: {str(e)}"

@tool
def generate_slide_image(prompt: str, output_file: str = "slide_image.png") -> str:
    """
    Generates a custom image/illustration using Bedrock Stable Image Ultra model in us-west-2,
    and saves it to the specified output file path. Uses a global lock and sequential cooldown
    delay to strictly prevent parallel request throttling.
    Falls back to a local Pillow-based vector generator if model access fails.
    """
    from PIL import Image, ImageDraw, ImageFont
    import time
    
    max_retries = 3
    retry_delay = 5
    
    # Serialize image generation calls via global thread lock
    with image_lock:
        print(f"DEBUG: Acquired lock. Processing image: '{prompt[:50]}...'")
        for attempt in range(max_retries + 1):
            try:
                print(f"DEBUG: Attempting to call invoke_model for stability.stable-image-ultra-v1:1 in us-west-2...")
                client = boto3.client(service_name="bedrock-runtime", region_name="us-west-2")
                request_body = {
                    "prompt": prompt,
                    "mode": "text-to-image",
                    "aspect_ratio": "16:9"
                }
                response = client.invoke_model(
                    modelId="stability.stable-image-ultra-v1:1",
                    body=json.dumps(request_body)
                )
                output_body = json.loads(response["body"].read().decode("utf-8"))
                base64_image_data = output_body["images"][0]
                image_data = base64.b64decode(base64_image_data)
                
                out_dir = os.path.dirname(os.path.abspath(output_file))
                if out_dir:
                    os.makedirs(out_dir, exist_ok=True)
                with open(output_file, "wb") as f:
                    f.write(image_data)
                print(f"DEBUG: Successfully generated image via Bedrock (Stable Image Ultra) and saved at: '{output_file}'")
                
                # Introduce a 4.5-second cooldown delay to stay safely within Bedrock rate limits (TPS)
                print("DEBUG: Cooling down for 4.5 seconds...")
                time.sleep(4.5)
                return f"Successfully generated image and saved at: {os.path.abspath(output_file)}"
            except Exception as bedrock_err:
                err_str = str(bedrock_err)
                if "ThrottlingException" in err_str or "TooManyRequestsException" in err_str or "429" in err_str:
                    if attempt < max_retries:
                        print(f"DEBUG: Throttled by Bedrock. Retrying in {retry_delay} seconds (attempt {attempt+1}/{max_retries})...")
                        time.sleep(retry_delay)
                        retry_delay *= 2
                        continue
                print(f"DEBUG: Bedrock invoke_model error: {bedrock_err}. Activating Pillow fallback...")
                break
        
    # 2. Fallback: Generate a high-quality dark-theme visual concept schematic using PIL
    try:
        width, height = 1024, 576
        img = Image.new("RGB", (width, height), color=(22, 30, 49)) # Lighter Slate Blue (CARD_BG)
        draw = ImageDraw.Draw(img)
        
        # Draw neon cyan border
        draw.rectangle([15, 15, width-15, height-15], outline=(0, 242, 254), width=3)
        
        # Draw abstract nodes representing biological cell/neural network connectivity
        draw.ellipse([150, 180, 310, 340], outline=(138, 43, 226), width=4)
        draw.ellipse([270, 240, 430, 400], outline=(0, 242, 254), width=4)
        draw.line([230, 260, 500, 260], fill=(148, 163, 184), width=2)
        draw.line([500, 260, 600, 360], fill=(148, 163, 184), width=2)
        draw.rectangle([600, 320, 860, 460], outline=(138, 43, 226), width=3)
        
        # Load font
        font_title = None
        font_body = None
        for font_path in [
            "/System/Library/Fonts/Supplemental/Arial.ttf",
            "/Library/Fonts/Arial.ttf",
            "Arial.ttf",
            "arial.ttf"
        ]:
            try:
                font_title = ImageFont.truetype(font_path, 28)
                font_body = ImageFont.truetype(font_path, 16)
                break
            except Exception:
                continue
        if font_title is None:
            font_title = ImageFont.load_default()
            font_body = ImageFont.load_default()
            
        # Parse prompt to a short title for display
        words = prompt.split()
        short_title = " ".join(words[:6]) + "..." if len(words) > 6 else prompt
        
        draw.text((100, 50), "VISUAL CONCEPT DIAGRAM", fill=(0, 242, 254), font=font_body)
        draw.text((100, 80), short_title.upper(), fill=(248, 250, 252), font=font_title)
        draw.text((620, 340), "CONCEPT ILLUSTRATION", fill=(248, 250, 252), font=font_body)
        draw.text((620, 370), "Dynamic placeholder graphic generated", fill=(148, 163, 184), font=font_body)
        draw.text((620, 395), "to support visual learning.", fill=(148, 163, 184), font=font_body)
        
        # Save output
        out_dir = os.path.dirname(os.path.abspath(output_file))
        if out_dir:
            os.makedirs(out_dir, exist_ok=True)
        img.save(output_file)
        
        print(f"DEBUG: Successfully generated Pillow fallback image and saved at: '{output_file}'")
        return f"Successfully generated image and saved at: {os.path.abspath(output_file)}"
    except Exception as pil_err:
        print(f"DEBUG ERROR: Pillow fallback failed: {pil_err}")
        return f"Failed to generate image. Error: {str(pil_err)}"

print("Slide and Image tools initialized.")


Slide and Image tools initialized.


## Step 3: Load Presentation Design Skills

We read and load the formatting and design guidelines from the project's skill document `skills/ppt_design/SKILL.md` directly into a Python string.

In [3]:
skill_path = "skills/ppt_design/SKILL.md"
if not os.path.exists(skill_path):
    skill_path = "experiment/skills/ppt_design/SKILL.md"

if os.path.exists(skill_path):
    with open(skill_path, "r", encoding="utf-8") as f:
        ppt_skill_content = f.read()
    print(f"Loaded {len(ppt_skill_content)} characters of guidelines from SKILL.md.")
else:
    ppt_skill_content = "Design slide outlines using widescreen (16:9) ratios and dark-theme configurations."
    print("Warning: SKILL.md not found. Loaded fallback instructions.")


Loaded 3238 characters of guidelines from SKILL.md.


## Step 4: Configure Bedrock Chat Model & Research Tool

We configure the LLM `us.anthropic.claude-haiku-4-5-20251001-v1:0` via Bedrock. We also load the `DuckDuckGoSearchRun` tool from `langchain-community` for web research.

In [4]:
from langchain_aws import ChatBedrockConverse
from langchain_community.tools import DuckDuckGoSearchRun

MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
AWS_REGION = "us-east-1"

llm = ChatBedrockConverse(
    model=MODEL_ID,
    temperature=0.7,
    region_name=AWS_REGION
)

# Initialize web search tool
search_tool = DuckDuckGoSearchRun()

print("LLM and Web Search tool ready.")

LLM and Web Search tool ready.


/var/folders/ry/0dp2gqbj4nv8q6qbdj7z67700000gn/T/ipykernel_23188/3261629026.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


## Step 5: Compile Agent with create_agent and MemorySaver

We construct the agent using the `create_agent` function and configure a `MemorySaver` checkpointer to store chat thread history across invocation cycles.

In [5]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

# State persistence checkpointer
memory = MemorySaver()

# Construct system prompt injecting our loaded design skill
system_prompt_str = (
    "You are an expert presentation designer, researcher, and architectural assistant. "
    "Your task is to help the user create a highly professional, well-structured, dark-theme PowerPoint presentation.\n\n"
    "LOADED DESIGN SKILLS & GUIDELINES:\n"
    "---------------------------------------\n"
    f"{ppt_skill_content}\n"
    "---------------------------------------\n\n"
    "CRITICAL WORKFLOW & TOOL CALLING RULES:\n"
    "1. CONDUCT RESEARCH FIRST: Always start by using the duckduckgo_search tool to research the topic. Gather structured facts, detailed definitions, and names.\n"
    "2. DRAFT THE OUTLINE: Plan the slides (Title slide, Content slides, and Ending slide). For EACH content slide, you should include a relevant visual illustration to make the presentation professional.\n"
    "3. GENERATE IMAGES BEFORE CREATING PRESENTATION: For every content slide, you MUST first call the 'generate_slide_image' tool to generate a relevant, high-quality diagram or illustration (e.g. save to 'experiment/slide2_concept.png'). You must do this for ALL slide images before proceeding to the next step.\n"
    "4. COMPILE THE PRESENTATION LAST: Once (and only when) all custom image files have been generated and saved to disk, call 'create_premium_presentation'. For each content slide, specify 'layout': 'image' and pass the correct local 'image_path' (e.g., 'experiment/slide2_concept.png') so the tool can embed it. If you call 'create_premium_presentation' before the images are created, they will not appear on the slides.\n"
    "5. CONFIRMATION: Confirm to the user the outline and where the final PPTX is saved.\n"
)

# Define tools list
agent_tools = [create_premium_presentation, generate_slide_image, search_tool]

# Compile Agent using the requested structured layout
ppt_agent = create_agent(
    model=llm,
    tools=agent_tools,
    system_prompt=system_prompt_str,
    checkpointer=memory
)

print("PPT Research & Design Agent compiled successfully.")

PPT Research & Design Agent compiled successfully.


## Step 6: Test the Multi-Tool Presentation Agent

We trigger the agent execution via `.invoke()`. We pass a thread configuration in `config` to enable the checkpointer.

In [6]:
# Setup execution configuration with thread_id for checkpointing
config = {"configurable": {"thread_id": "presentation-thread-02"}}

topic_prompt = """Create a professional 10-slide presentation on Python Programming. Make sure to research the topic first using web search, generate custom images for each content slide using the image generation tool, and then compile the presentation with the correct image paths. The presentation should be designed for an audience of beginner to intermediate programmers, and should cover key concepts, popular libraries, and best practices in Python programming."""


try:
    print(f"Sending prompt to Agent:\n'{topic_prompt}'\n")
    response = ppt_agent.invoke(
        {"messages": [("user", topic_prompt)]},
        config=config
    )
    
    # Display result
    final_response = response["messages"][-1].content
    print("\n--- Agent Response ---")
    print(final_response)
except Exception as e:
    print(f"An error occurred during agent run: {e}")

Sending prompt to Agent:
'Create a professional 10-slide presentation on Python Programming. Make sure to research the topic first using web search, generate custom images for each content slide using the image generation tool, and then compile the presentation with the correct image paths. The presentation should be designed for an audience of beginner to intermediate programmers, and should cover key concepts, popular libraries, and best practices in Python programming.'

DEBUG: Acquired lock. Processing image: 'Python functions and object-oriented programming c...'
DEBUG: Attempting to call invoke_model for stability.stable-image-ultra-v1:1 in us-west-2...
DEBUG: Successfully generated image via Bedrock (Stable Image Ultra) and saved at: 'experiment/slide4_functions_oop.png'
DEBUG: Cooling down for 4.5 seconds...
DEBUG: Acquired lock. Processing image: 'Python data structures visualization: lists, tuple...'
DEBUG: Attempting to call invoke_model for stability.stable-image-ultra-v1:1